In [ ]:
%%writefile fmm3d.h
#pragma once           // il file viene incluso una sola volta
#include <cstdio>    
#include <vector>
#include <cmath>      // fminf, sqrtf
#include <queue>       // std::priority_queue
#include <functional>  // std::greater
#include <utility>     // std::pair
#include <chrono>      // misura dei tempi
#include <fstream>     // scrittura su file
const float INF = 1e30f;
typedef std::pair<float, int> Entry; // coppia (u, indice) per la coda di priorita'

__host__ __device__ float solve_quadratic3(float a, float b, float c,float F, float h)
{
    float Fh = F * h;
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}

float solve_node(const std::vector<float>& u, const std::vector<float>& F,
                 int k, int i, int j, int N, float h)
{
    int idx = (k * N + i) * N + j;  // indice globale del nodo (k, i, j)
    // a: min dei vicini lungo x (stessa riga, colonna j-1 e j+1)
    float left  = (j > 0)     ? u[idx - 1] : INF;
    float right = (j < N - 1) ? u[idx + 1] : INF;
    // b: min dei vicini lungo y (stessa colonna, riga i-1 e i+1)
    float down  = (i > 0)     ? u[idx - N] : INF;
    float up    = (i < N - 1) ? u[idx + N] : INF;
    // c: min dei vicini lungo z (stessa colonna e riga, piano k-1 e k+1)
    float back  = (k > 0)     ? u[(idx - N * N)] : INF;
    float front = (k < N - 1) ? u[(idx + N *N)] : INF;

    // Per arrivare devi scorrere come un indice globale: u[i][j][k] = u[(k * N + i) * N + j]
    float a = fminf(left, right);
    float b = fminf(down, up);
    float c = fminf(back, front);

    float q = solve_quadratic3(a, b, c, F[idx], h);
    return fminf(u[idx], q);                                  // return min(u_ijk, u)
}
void fmm(std::vector<float>& u, const std::vector<float>& F, int N, float h, int src)
{
    enum { LONTANO = 0, ATTIVO = 1, NOTO = 2 };      // stato di ogni nodo
    std::vector<char> state(N * N * N, LONTANO);    // array di char NxNxN, inizializzato a LONTANO

    std::priority_queue<Entry, std::vector<Entry>, std::greater<Entry>> heap;
    heap.push(Entry(u[src], src));                   // la sorgente e' il primo Trial
    state[src] = ATTIVO;

    const int dk[6] = { 0, 0, 0, 0, -1, 1 };              // spostamenti verso i 6 vicini:
    const int di[6] = {  0, 0, -1, 1, 0, 0 };              // spostamenti verso i 6 vicini:
    const int dj[6] = { -1, 1,  0, 0, 0, 0 };              // sinistra, destra, sotto, sopra

    while (!heap.empty()) {                          // while A != vuoto
        Entry trial = heap.top();                    // Trial <- radice del min-heap
        heap.pop();                                  // Rimuovi Trial da A
        int x = trial.second;                        // indice del nodo Trial
        if (state[x] == NOTO || trial.first > u[x])  // coppia "scaduta": si ignora
            continue;
        state[x] = NOTO;                             // Aggiungi Trial a Set

        int k = x / (N*N), i = (x / N) % N, j = x % N;                    // strato, riga, colonna del nodo Trial
        for (int d = 0; d < 6; ++d) {                // foreach x_nb adiacente a Trial
            int ni = i + di[d], nj = j + dj[d], nk = k + dk[d];
            if (ni < 0 || ni >= N || nj < 0 || nj >= N || nk < 0 || nk >= N) continue;   // fuori dal dominio
            int nb = (nk * N + ni) * N + nj;   // indice del nodo vicino
            if (state[nb] == NOTO) continue;         // if x_nb non appartiene a Set

            float q = solve_node(u, F, nk, ni, nj, N, h);
            if (q < u[nb]) {                         // u(x_nb) <- soluzione di g(x_nb)
                u[nb] = q;
                state[nb] = ATTIVO;                  // Aggiungi x_nb ad A
                heap.push(Entry(q, nb));
            }
        }
    }
}
double relative_error(const std::vector<float>& u, int N, float h)
{
    double num = 0.0, den = 0.0;
    for (int k = 0; k < N; ++k)
        for (int i = 0; i < N; ++i)
            for (int j = 0; j < N; ++j) {
            double x = -1.0 + j * h, y = -1.0 + i * h, z = -1.0 + k * h;   // coordinate del nodo (k, i, j)
            double U = std::sqrt(x * x + y * y + z * z);        // soluzione esatta
            double d = u[(k * N + i) * N + j] - U;
            num += d * d;                               // ||u - U||^2
            den += U * U;                               // ||U||^2
            }
    return std::sqrt(num / den);    // Norma euclidea relativa: ||u - U|| / ||U||
}


In [ ]:
%%writefile fim3d.cu
#include <cstdio>          
#include <cstdlib>         
#include <cuda_runtime.h>
#include <algorithm>       // std::sort
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
#include "fmm3d.h"           // Fast Marching

// Controllo degli errori CUDA: se la chiamata fallisce, stampa file, riga e motivo ed esce.
#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

#define BLOCK 8  // dimensione del blocco 8x8x8 = 512 thread
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}

__device__ void process_block(int block, int niter, float* u, const float* F,
                             int N, int nbx, float h, int src, float eps, unsigned char* Cb_out)
{
    __shared__ float s_u[BLOCK + 2][BLOCK + 2][BLOCK + 2];                // block + halo, nella shared memory
    __shared__ unsigned char s_c[BLOCK * BLOCK * BLOCK];               // i flag Cn dei 512 nodi

    int bi = (block / nbx) % nbx, bj = block % nbx, bk = block / (nbx * nbx);                    // riga, colonna e strato del blocco
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    int tid = (tz * BLOCK + ty) * BLOCK + tx;                                // numero del thread nel blocco: 0..511
    int i = bi * BLOCK + ty, j = bj * BLOCK + tx, k = bk * BLOCK + tz;              // nodo di questo thread
    int idx = (k * N + i) * N + j;  // posizione del nodo nella griglia
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || (idx == src);             // nodi da non aggiornare
    float f = inside ? F[idx] : 1.0f;                  // F_ij del nodo

    // Global -> Shared: 1000 celle (block + halo), 512 thread
    for (int m = tid; m < (BLOCK + 2) * (BLOCK + 2) * (BLOCK + 2); m += BLOCK * BLOCK * BLOCK) {
        int sz = m / ((BLOCK + 2) * (BLOCK + 2));
        int sy = (m / (BLOCK + 2)) % (BLOCK + 2);
        int sx = m % (BLOCK + 2);       // cella (sz, sy, sx) della shared memory
        int gi = bi * BLOCK + sy - 1;                         // nodo della griglia corrispondente:
        int gj = bj * BLOCK + sx - 1;                         
        int gk = bk * BLOCK + sz - 1;                         // -1 perche' lo strato/riga/colonna 0 e' l'halo
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();                                         // la block e' completa prima di usarla

    // niter iterazioni locali in shared memory
    bool conv = true;                                        // Cn del nodo
    for (int it = 0; it < niter; ++it) {
        float p = s_u[tz+1][ty + 1][tx + 1];                       // valore prima dell'aggiornamento
        float q = fixed ? p                                  // sorgente / fuori: non cambia
                        : solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        __syncthreads();                                     // (A) tutti hanno letto i vicini...
        s_u[tz+1][ty + 1][tx + 1] = q;
        __syncthreads();                                     // (B) ...e tutti hanno scritto
        conv = fabsf(p - q) < eps;                           // Cn: il nodo non e' cambiato
    }

    // Shared -> Global: ogni thread riscrive il PROPRIO nodo (non l'halo)
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];

    // Cb(b) <- reduction(Cn(b)): AND dei 64 flag
    s_c[tid] = conv;
    __syncthreads();
    for (int stride = BLOCK * BLOCK * BLOCK / 2; stride > 0; stride /= 2) {     // 256, ..., 4, 2, 1
        if (tid < stride) s_c[tid] = s_c[tid] && s_c[tid + stride];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_c[0];                     // vero solo se TUTTI i nodi sono convergenti
}

__global__ void update_active_blocks(const int* list, float* u, const float* F,
                                    int N, int nbx, float h, int src, int niter,
                                    float eps, unsigned char* Cb1)
{
    process_block(list[blockIdx.x], niter, u, F, N, nbx, h, src, eps, Cb1);
}

__global__ void build_active_list(const unsigned char* Cb, unsigned char* active,
                                  int* list, int* count, int n_blocks) 
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // numero del blocco di nodi
    if (t >= n_blocks) return;                                // thread in eccesso
    bool act = !Cb[t];                                       // attivo se NON convergente
    active[t] = act;
    if (act) list[atomicAdd(count, 1)] = t;                  // aggiungi t alla lista A
}

__global__ void build_candidate_list(const int* list, int count, int nbx, const unsigned char* active,
                                     const unsigned char* Cb1, unsigned char* Cb,
                                     int* cand, int* cand_list, int* cand_count)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;
    if (t >= count) return;                                  // thread in eccesso
    int b = list[t];                                         // un blocco della lista A
    Cb[b] = Cb1[b];                                          // non convergente (Caso 1): resta in A
    if (!Cb1[b]) return;                                     // proseguono solo i blocchi appena convergenti

    int bk = b / (nbx * nbx), bi = (b / nbx) % nbx, bj = b % nbx;                          // strato, riga e colonna del blocco
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};  // sinistra, destra, sotto, sopra e davanti, dietro
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // fuori dalla griglia di blocchi
        if (active[nb[d]] && !Cb1[nb[d]]) continue;          // Caso 1: lo aggiorna già lo Step 1
        if (atomicExch(&cand[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}

__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* cand,
                                        float* u, const float* F, int N, int nbx, float h,
                                        int src, float eps, unsigned char* Cb)
{
    if (blockIdx.x >= *cand_count) return;                   // blocco CUDA in eccesso
    int b = cand_list[blockIdx.x];
    process_block(b, 1, u, F, N, nbx, h, src, eps, Cb);      // test del blocco: 1 iterazione
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) cand[b] = 0;   // pronto per il giro successivo
}

int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              int src, float eps, int niter, float* ms)
{
    const int nbx = (N + BLOCK - 1) / BLOCK, n_blocks = nbx * nbx * nbx; // nbx = numero di blocchi per riga/colonna, n_blocks = numero totale di blocchi
    const size_t bytes = (size_t)N * N * N * sizeof(float); // dimensione in byte della griglia

    std::vector<unsigned char> Cb0(n_blocks, 1);
    const int sk = src / (N * N), si = (src / N) % N, sj = src % N;   // strato, riga e colonna del nodo sorgente
    Cb0[((sk / BLOCK) * nbx + si % BLOCK)* nbx + sj/BLOCK] = 0;   // vale 1 ovunque tranne che nel blocco della sorgente

    // Memoria sulla GPU e dati iniziali
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb1, *d_active; int *d_list, *d_count;
    int *d_cand, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_u, bytes));
    CUDA_CHECK(cudaMalloc(&d_F, bytes));
    CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb1, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_active, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int)));

    // Allocate memory for candidate blocks
    CUDA_CHECK(cudaMalloc(&d_cand, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));

    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb0.data(), n_blocks, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemset(d_Cb1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_cand, 0, n_blocks * sizeof(int)));


    const int threads = 256, list_blocks = (n_blocks + threads - 1) / threads;   // numero di blocchi per costruire la lista A
    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                      // "segna l'ora" prima del ciclo
    int iter = 0;
    while (true) {
        int count = 0;
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));
        build_active_list<<<list_blocks, threads>>>(d_Cb, d_active, d_list, d_count, n_blocks);
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(&count, d_count, sizeof(int), cudaMemcpyDeviceToHost));
        if (count == 0) break;
        ++iter;

        // Step 1: foreach b in A in paralleli attivi
        update_active_blocks<<<count, dim3(BLOCK, BLOCK, BLOCK)>>>(d_list, d_u, d_F, N, nbx, h, src, niter, eps, d_Cb1);
        CUDA_CHECK(cudaGetLastError());

        // Step 2: solo i vicini dei blocchi appena convergenti
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));
        build_candidate_list<<<(count + threads - 1) / threads, threads>>>(d_list, count, nbx, d_active, d_Cb1, d_Cb,
                                                                          d_cand, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());
        update_candidate_blocks<<<std::min(6 * count, n_blocks), dim3(BLOCK, BLOCK, BLOCK)>>>(d_cand_list, d_cand_count, d_cand,
                                                                                    d_u, d_F, N, nbx, h, src, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());
    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));
    CUDA_CHECK(cudaEventDestroy(stop));

    // Risultato sull'host e pulizia
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost));
    CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb1));
    CUDA_CHECK(cudaFree(d_active)); CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_count));
    CUDA_CHECK(cudaFree(d_cand)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;
}

int main()
{
    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s | multiprocessori: %d | max thread per blocco: %d | shared memory per blocco: %zu KB\n",
           prop.name, prop.multiProcessorCount, prop.maxThreadsPerBlock, prop.sharedMemPerBlock / 1024);
    
    CUDA_CHECK(cudaFree(0));                                 // inizializza la GPU prima delle misure
    const float eps = 1e-6f;                                 // tolleranza di convergenza
    const int niter = 3 * BLOCK;                              // 24 iterazioni locali
    const int ripetizioni = 3;                               // ogni tempo = il minimo di 3 misure
    printf("\n    N   iter glob.   max|FIM - FMM|   errore FIM   t FMM [ms]   t FIM [ms]   speedup\n");
    for (int Nk : { 33, 65, 129, 193, 257 }) {
        const float hk = 2.0f / (Nk - 1);                    // griglia (§2.4)
        const int srck = ((Nk / 2) * Nk + Nk / 2) * Nk + (Nk / 2);             // sorgente al centro (eq. 4.4)
        std::vector<float> Fk(Nk * Nk * Nk, 1.0f);                // mezzo omogeneo
        std::vector<float> u0(Nk * Nk * Nk, INF);                 // condizione iniziale
        u0[srck] = 0.0f;

        std::vector<float> uk_fmm, uk_fim;
        double t_fmm = 1e30, t_fim = 1e30;
        int it = 0;
        for (int r = 0; r < ripetizioni; ++r) {
            uk_fmm = u0;                                     // ogni misura riparte da capo
            auto t0 = std::chrono::high_resolution_clock::now();
            fmm(uk_fmm, Fk, Nk, hk, srck);                   // CPU: Algoritmo 3
            auto t1 = std::chrono::high_resolution_clock::now();
            t_fmm = std::min(t_fmm, std::chrono::duration<double, std::milli>(t1 - t0).count());

            uk_fim = u0;
            float ms = 0.0f;
            it = block_fim(uk_fim, Fk, Nk, hk, srck, eps, niter, &ms);   // GPU: Algoritmo 7
            t_fim = std::min(t_fim, (double)ms);
        }
        
        float diff = 0.0f;
        for (int k = 0; k < Nk * Nk * Nk; ++k) diff = fmaxf(diff, fabsf(uk_fim[k] - uk_fmm[k]));
        printf("%5d  %9d   %14.2e   %10.3e   %10.1f   %10.2f   %7.1f\n", Nk, it, diff,
               relative_error(uk_fim, Nk, hk), t_fmm, t_fim, t_fmm / t_fim);

        if (Nk == 65) {                                     // soluzione FIM per il grafico (Python)
            std::ofstream out("fim_65.bin", std::ios::binary);
            out.write(reinterpret_cast<const char*>(uk_fim.data()), uk_fim.size() * sizeof(float));
        }
    }

    return 0;
}


In [ ]:
!nvcc -O3 -arch=sm_75 fim3d.cu -o fim3d && ./fim3d